# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.### Dataset SourceThe dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install -U mlcroissant

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport pprint# Define the dataset URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'# Load the dataset metadatadataset = mlc.Dataset(croissant_url)print(f"Dataset Name: {dataset.metadata.name}")print(f"Description: {dataset.metadata.description}")

## 2. Data OverviewReview available record sets, their fields, and list their IDs.

In [ ]:
# List all record sets by @id and nameprint("Available Record Sets:")record_sets = [rs for rs in dataset.metadata.record_sets]for rs in record_sets:    print(f"@id: {rs['@id']}, name: {rs['name']}")# For demonstration, display fields for each record setprint("\nFields for each Record Set:")for rs in record_sets:    print(f"\nRecord Set {rs['name']} (@id: {rs['@id']}):")    for field in rs['fields']:        print(f"- Field @id: {field['@id']}, name: {field['name']}, dataType: {field['dataType']}")

## 3. Data ExtractionLoad data from the main protein record set into a DataFrame for analysis. Use the `@id` from the overview above.

In [ ]:
# Pick the main record set (@id) associated with protein records# The actual @id must be copy-pasted from above for accuracy.# For instance, assume one record set has @id 'cr:ProteinRecords' (replace with real value if different).protein_record_set_id = Nonefor rs in record_sets:    if 'protein' in rs['name'].lower() or 'main' in rs['name'].lower():        protein_record_set_id = rs['@id']        breakif protein_record_set_id is None:    # Fallback to first record set    protein_record_set_id = record_sets[0]['@id']all_record_set_ids = [rs['@id'] for rs in record_sets]dataframes = {}for record_set_id in all_record_set_ids:    # Some record sets might be empty or huge. Use try/except to load all.    try:        records = list(dataset.records(record_set=record_set_id))        if records:            dataframes[record_set_id] = pd.DataFrame(records)            print(f"Loaded {len(records)} records from record set {record_set_id}")        else:            print(f"No records for record set {record_set_id}")    except Exception as e:        print(f"Error loading record set {record_set_id}: {str(e)}")if protein_record_set_id in dataframes:    print('Columns in DataFrame:', dataframes[protein_record_set_id].columns.tolist())    dataframes[protein_record_set_id].head()else:    print(f"No DataFrame loaded for {protein_record_set_id}")

## 4. Exploratory Data Analysis (EDA)Apply common data processing: filter records, normalize a numeric field, and group data by class. This helps prepare for detailed scientific and machine learning analysis.**Note:** Replace `field_id_numeric` and `field_id_class` with actual `@id` values from the overview above. We'll pick the first numeric and first categorical field.

In [ ]:
df = dataframes[protein_record_set_id]# Identify numeric and categorical field candidates (by checking dtype and by Croissant field definitions)numeric_field = Noneclass_field = None# Reuse metadata to get Croissant dataType info (since pandas types might not match for all fields)rs_meta = Nonefor rs in record_sets:    if rs['@id'] == protein_record_set_id:        rs_meta = rs        breakif rs_meta:    for field in rs_meta['fields']:        if not numeric_field and field.get('dataType') in ['schema:Number', 'schema:Float', 'schema:Integer'] and field['@id'] in df.columns:            numeric_field = field['@id']        if not class_field and field.get('dataType') == 'schema:Text' and field['@id'] in df.columns:            class_field = field['@id']if not numeric_field:    # Fallback: try first float or int column in dataframe    for col in df.columns:        if pd.api.types.is_numeric_dtype(df[col]):            numeric_field = col            breakif not class_field:    for col in df.columns:        if pd.api.types.is_object_dtype(df[col]):            class_field = col            breakprint(f"Using numeric field: {numeric_field}")print(f"Using class/grouping field: {class_field}")# Filter for records with field > thresholdthreshold = 10df_filter = df[df[numeric_field].astype(float) > threshold]print(f"Filtered records with {numeric_field} > {threshold} (show first 5):")print(df_filter[[numeric_field]].head())# Normalize numeric fielddf_filter[f"{numeric_field}_normalized"] = (df_filter[numeric_field].astype(float) - df_filter[numeric_field].astype(float).mean()) / df_filter[numeric_field].astype(float).std()print(f"\nNormalized {numeric_field} (first 5):")print(df_filter[[numeric_field, f"{numeric_field}_normalized"]].head())# Group and aggregate (mean) by class field if presentif class_field in df_filter.columns:    grouped_df = df_filter.groupby(class_field)[numeric_field].mean().reset_index()    print(f"\nGrouped mean {numeric_field} by {class_field} (first 5):")    print(grouped_df.head())

## 5. VisualizationVisualize the distribution of the selected numeric field and its grouping, if available.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns# Histogram of numeric fieldplt.figure(figsize=(8, 5))sns.histplot(df[numeric_field].astype(float), bins=30)plt.xlabel(numeric_field)plt.title(f"Distribution of {numeric_field}")plt.show()# Boxplot by class field if availableif class_field in df.columns:    plt.figure(figsize=(12, 6))    sns.boxplot(data=df, x=class_field, y=numeric_field)    plt.xticks(rotation=45, ha='right')    plt.title(f"{numeric_field} by {class_field}")    plt.show()

## 6. ConclusionIn this notebook, we loaded the FAIR² mass spectrometry dataset using its Croissant schema, inspected metadata, explored record sets and fields by their `@id`, and performed basic EDA including filtering, normalization, aggregation, and visualization using both Croissant and pandas conventions.The above steps can be adapted to any Croissant-compatible dataset by substituting the schema URL and using the correct record set and field `@id`s for your analysis.